In [ ]:
import os
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# =========================================================
# Config
# =========================================================
CSV_PATH = "data_index.csv"
CHECKPOINT_DIR = "."                 # where best_model_fold*.pth are stored
DATA_ROOT = "."                      # root prefix if file_path in CSV is relative
BATCH_SIZE = 32
NUM_WORKERS = 0                      # set >0 if you want
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

N_SPLITS = 5
RANDOM_STATE = 42

CLASS_NAMES = ["Asthma", "COPD", "ILD", "Infection", "Healthy"]
MAX_LEN = 512
IN_CHANNELS = 40
N_CLASSES = 5

# Output files
OUT_DIR = Path("fold_metrics_outputs")
OUT_DIR.mkdir(exist_ok=True)

FOLD_SUMMARY_CSV = OUT_DIR / "fold_summary_metrics.csv"
PER_CLASS_CSV = OUT_DIR / "per_class_metrics.csv"
CONFUSION_CSV = OUT_DIR / "confusion_matrices_long.csv"


# =========================================================
# Model
# =========================================================
class TemporalAttentionPooling(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.attn = nn.Linear(in_features, 1)

    def forward(self, x):
        """
        x: (B, C, T)
        returns: (B, C)
        """
        # transpose -> (B, T, C)
        x_t = x.transpose(1, 2)
        scores = self.attn(x_t)                  # (B, T, 1)
        weights = torch.softmax(scores, dim=1)   # (B, T, 1)
        pooled = torch.sum(weights * x_t, dim=1) # (B, C)
        return pooled


class CNN1DAttention(nn.Module):
    def __init__(self, in_channels=40, n_classes=5, dropout=0.5):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
        )

        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
        )

        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
        )

        self.attention = TemporalAttentionPooling(256)

        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        """
        x: (B, 40, 512)
        """
        x = self.conv1(x)   # -> (B, 64, 256)
        x = self.conv2(x)   # -> (B, 128, 128)
        x = self.conv3(x)   # -> (B, 256, 64)
        x = self.attention(x)
        x = self.fc(x)
        return x


# =========================================================
# Dataset
# =========================================================
class SpectrogramDataset(Dataset):
    def __init__(self, dataframe, label_to_idx, data_root=".", max_len=512, in_channels=40):
        self.df = dataframe.reset_index(drop=True).copy()
        self.label_to_idx = label_to_idx
        self.data_root = Path(data_root)
        self.max_len = max_len
        self.in_channels = in_channels

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, rel_or_abs_path: str) -> Path:
        p = Path(rel_or_abs_path)
        if p.is_absolute():
            return p
        return self.data_root / p

    def _fix_shape(self, arr: np.ndarray) -> np.ndarray:
        """
        Expect final shape (40, 512)
        """
        if arr.ndim != 2:
            raise ValueError(f"Expected 2D array, got shape {arr.shape}")

        # If shape is (T, 40), transpose it
        if arr.shape[0] != self.in_channels and arr.shape[1] == self.in_channels:
            arr = arr.T

        if arr.shape[0] != self.in_channels:
            raise ValueError(
                f"Expected first dim = {self.in_channels}, got {arr.shape}. "
                f"Check MFCC saved shape."
            )

        T = arr.shape[1]

        if T < self.max_len:
            pad_width = self.max_len - T
            arr = np.pad(arr, ((0, 0), (0, pad_width)), mode="constant")
        elif T > self.max_len:
            arr = arr[:, :self.max_len]

        return arr.astype(np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = self._resolve_path(row["file_path"])
        label_name = row["Diagnosis"]

        arr = np.load(file_path)
        arr = self._fix_shape(arr)

        x = torch.tensor(arr, dtype=torch.float32)
        y = torch.tensor(self.label_to_idx[label_name], dtype=torch.long)

        return x, y, str(file_path)


# =========================================================
# Metrics
# =========================================================
def compute_multiclass_specificity(cm: np.ndarray):
    """
    cm: confusion matrix of shape (C, C)
    Returns per-class specificity and macro specificity.
    """
    n_classes = cm.shape[0]
    total = cm.sum()

    per_class_specificity = []

    for i in range(n_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - (tp + fn + fp)

        denom = tn + fp
        spec = tn / denom if denom > 0 else 0.0
        per_class_specificity.append(spec)

    per_class_specificity = np.array(per_class_specificity, dtype=np.float64)
    macro_specificity = float(np.mean(per_class_specificity))

    return per_class_specificity, macro_specificity


def evaluate_fold(model, loader, device, class_names):
    model.eval()

    all_true = []
    all_pred = []

    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            preds = torch.argmax(logits, dim=1)

            all_true.extend(y.cpu().numpy().tolist())
            all_pred.extend(preds.cpu().numpy().tolist())

    all_true = np.array(all_true)
    all_pred = np.array(all_pred)

    cm = confusion_matrix(all_true, all_pred, labels=list(range(len(class_names))))

    acc = accuracy_score(all_true, all_pred)
    macro_f1 = f1_score(all_true, all_pred, average="macro", zero_division=0)

    precision, recall, f1, support = precision_recall_fscore_support(
        all_true,
        all_pred,
        labels=list(range(len(class_names))),
        average=None,
        zero_division=0,
    )

    macro_precision = float(np.mean(precision))
    macro_recall = float(np.mean(recall))  # sensitivity
    per_class_specificity, macro_specificity = compute_multiclass_specificity(cm)

    metrics = {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "macro_precision": float(macro_precision),
        "macro_sensitivity": float(macro_recall),
        "macro_specificity": float(macro_specificity),
        "confusion_matrix": cm,
        "per_class": [],
    }

    for i, cls_name in enumerate(class_names):
        metrics["per_class"].append(
            {
                "class_name": cls_name,
                "precision": float(precision[i]),
                "recall": float(recall[i]),
                "f1": float(f1[i]),
                "specificity": float(per_class_specificity[i]),
                "support": int(support[i]),
            }
        )

    return metrics


# =========================================================
# Utilities
# =========================================================
def load_checkpoint(model, ckpt_path, device):
    checkpoint = torch.load(ckpt_path, map_location=device)

    # common patterns:
    # 1) raw state_dict
    # 2) {"model_state_dict": ...}
    # 3) {"state_dict": ...}
    if isinstance(checkpoint, dict):
        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        elif "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        else:
            # could already be a state_dict-like dict
            state_dict = checkpoint
    else:
        state_dict = checkpoint

    # remove "module." prefix if trained with DataParallel
    cleaned_state_dict = {}
    for k, v in state_dict.items():
        new_k = k.replace("module.", "") if k.startswith("module.") else k
        cleaned_state_dict[new_k] = v

    missing, unexpected = model.load_state_dict(cleaned_state_dict, strict=False)

    if missing:
        print(f"[WARN] Missing keys in {ckpt_path}: {missing}")
    if unexpected:
        print(f"[WARN] Unexpected keys in {ckpt_path}: {unexpected}")

    return model


def print_fold_confusion_matrix(cm, class_names, fold_idx):
    print(f"\nConfusion Matrix - Fold {fold_idx}")
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{c}" for c in class_names],
        columns=[f"pred_{c}" for c in class_names],
    )
    print(cm_df.to_string())


# =========================================================
# Main
# =========================================================
def main():
    print(f"Using device: {DEVICE}")

    df = pd.read_csv(CSV_PATH)

    required_cols = {"file_path", "Diagnosis", "SubjectID"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"CSV missing required columns: {missing_cols}")

    # Keep class order fixed
    unique_labels = df["Diagnosis"].unique().tolist()
    if set(CLASS_NAMES) != set(unique_labels):
        warnings.warn(
            f"CSV classes {sorted(unique_labels)} differ from CLASS_NAMES {sorted(CLASS_NAMES)}. "
            f"Proceeding with CLASS_NAMES order."
        )

    label_to_idx = {label: i for i, label in enumerate(CLASS_NAMES)}
    y_all = df["Diagnosis"].map(label_to_idx).values

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    fold_rows = []
    per_class_rows = []
    confusion_rows = []

    for fold_idx, (_, val_idx) in enumerate(skf.split(df, y_all)):
        print(f"\n{'=' * 60}")
        print(f"Evaluating Fold {fold_idx}")
        print(f"{'=' * 60}")

        val_df = df.iloc[val_idx].reset_index(drop=True)

        val_dataset = SpectrogramDataset(
            dataframe=val_df,
            label_to_idx=label_to_idx,
            data_root=DATA_ROOT,
            max_len=MAX_LEN,
            in_channels=IN_CHANNELS,
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
        )

        model = CNN1DAttention(in_channels=IN_CHANNELS, n_classes=N_CLASSES).to(DEVICE)

        ckpt_path = Path(CHECKPOINT_DIR) / f"best_model_fold{fold_idx}.pth"
        if not ckpt_path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

        model = load_checkpoint(model, ckpt_path, DEVICE)

        metrics = evaluate_fold(model, val_loader, DEVICE, CLASS_NAMES)

        # fold summary row
        fold_rows.append(
            {
                "fold": fold_idx,
                "num_val_samples": len(val_df),
                "accuracy": metrics["accuracy"],
                "macro_f1": metrics["macro_f1"],
                "macro_precision": metrics["macro_precision"],
                "macro_sensitivity": metrics["macro_sensitivity"],
                "macro_specificity": metrics["macro_specificity"],
            }
        )

        # per-class rows
        for cls_metrics in metrics["per_class"]:
            per_class_rows.append(
                {
                    "fold": fold_idx,
                    "class_name": cls_metrics["class_name"],
                    "precision": cls_metrics["precision"],
                    "recall": cls_metrics["recall"],
                    "f1": cls_metrics["f1"],
                    "specificity": cls_metrics["specificity"],
                    "support": cls_metrics["support"],
                }
            )

        # confusion matrix rows
        cm = metrics["confusion_matrix"]
        for i, true_cls in enumerate(CLASS_NAMES):
            for j, pred_cls in enumerate(CLASS_NAMES):
                confusion_rows.append(
                    {
                        "fold": fold_idx,
                        "true_class": true_cls,
                        "pred_class": pred_cls,
                        "count": int(cm[i, j]),
                    }
                )

        print_fold_confusion_matrix(cm, CLASS_NAMES, fold_idx)

        print("\nFold metrics:")
        print(f"  Accuracy          : {metrics['accuracy']:.4f}")
        print(f"  Macro F1          : {metrics['macro_f1']:.4f}")
        print(f"  Macro Sensitivity : {metrics['macro_sensitivity']:.4f}")
        print(f"  Macro Specificity : {metrics['macro_specificity']:.4f}")

    # =====================================================
    # Save CSVs
    # =====================================================
    fold_df = pd.DataFrame(fold_rows)
    per_class_df = pd.DataFrame(per_class_rows)
    confusion_df = pd.DataFrame(confusion_rows)

    fold_df.to_csv(FOLD_SUMMARY_CSV, index=False)
    per_class_df.to_csv(PER_CLASS_CSV, index=False)
    confusion_df.to_csv(CONFUSION_CSV, index=False)

    # =====================================================
    # Summary stats
    # =====================================================
    acc_mean = fold_df["accuracy"].mean()
    acc_std = fold_df["accuracy"].std(ddof=1)

    f1_mean = fold_df["macro_f1"].mean()
    f1_std = fold_df["macro_f1"].std(ddof=1)

    print(f"\n{'=' * 60}")
    print("Per-Fold Summary Table")
    print(f"{'=' * 60}")
    print(fold_df.to_string(index=False))

    print(f"\n{'=' * 60}")
    print("Cross-Fold Summary")
    print(f"{'=' * 60}")
    print(f"Accuracy : {acc_mean:.4f} ± {acc_std:.4f}")
    print(f"Macro F1 : {f1_mean:.4f} ± {f1_std:.4f}")

    summary_text = {
        "accuracy_mean": float(acc_mean),
        "accuracy_std": float(acc_std),
        "macro_f1_mean": float(f1_mean),
        "macro_f1_std": float(f1_std),
    }

    with open(OUT_DIR / "cross_fold_summary.json", "w") as f:
        json.dump(summary_text, f, indent=2)

    print(f"\nSaved files:")
    print(f"  {FOLD_SUMMARY_CSV}")
    print(f"  {PER_CLASS_CSV}")
    print(f"  {CONFUSION_CSV}")
    print(f"  {OUT_DIR / 'cross_fold_summary.json'}")


if __name__ == "__main__":
    main()